# Batch Preprocessing

**Author:** Noah Mba, noah.mba@fu-berlin.de  
**Date:** July 22, 2026  
**AI Acknowledgements:** Co-authored/Supported by Gemini and Claude 5 Sonnet  

---

This notebook runs the preprocessing pipeline for all subjects that should be included in the final analyses. It does so by defining a preprocessing function, that takes either a single or multiple subject IDs aswell as subject-specific configuration settings, as inputs.

**Update (November 3, 2026):** The pipeline is now split into two stages, so that the same cleaned continuous data can be epoched for different analyses. **Stage A** (`preprocess_continuous`) runs steps 1-4 once per subject and saves the cleaned continuous recording. **Stage B** (`make_epochs`) runs steps 5-7 (epoching, Autoreject, re-referencing) for a given epoch specification: `"erp"` reproduces the original ERP epochs, `"ers"` creates the cue-locked epochs for the encoding-retrieval similarity analyses (H5-H7). All steps, parameters and comments of the original notebook are kept.

### 1. Setup: Loading modules and objects, creating basic paths and helper function

This initialization cell prepares the workspace by loading the relevant modules and specifying the relevant paths and directories.

In [6]:
# ==========================================
# 1. IMPORTS & GLOBAL PATHS
# ==========================================
import mne
import pandas as pd
import numpy as np
import json
import os
from pathlib import Path
from mne_bids import BIDSPath, read_raw_bids
from autoreject import AutoReject
import matplotlib
matplotlib.use('Agg')  # No interactive pop-ups during batch runs
import matplotlib.pyplot as plt
mne.viz.set_browser_backend("matplotlib")

# ==========================================
# 2. Path Definitions & Global Variables 
# ==========================================
# This notebook should be located in project_folder/scripts/eeg
project_root = Path.cwd().parent.parent
bids_root = project_root / "data" / "bids"
derivatives_dir = project_root / "data" / "derivatives"
config_path = derivatives_dir / "preprocessing_config.csv"

# ==========================================
# 3. Load Configuration dataframe
# ==========================================
df_config = pd.read_csv(config_path, dtype={'subject': str})

# ==========================================
# 3. HELPER FUNCTION: SAVE A FIGURE SAFELY
# ==========================================
def save_fig(fig, out_dir, fname):
    """Save and close a figure, creating the directory if needed."""
    out_dir.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_dir / fname, dpi=100, bbox_inches='tight')
    plt.close(fig)

### 2. Define Preprocessing Pipeline as a customizable function

The preprocessing steps that are included and their order is mainly informed by the [**COBIDAS report** on Best Practices in Data Analysis and Sharing in Neuroimaging using MEEG (Pernet et al., 2018)](https://doi.org/10.1038/s41593-020-00709-0). We will adjust the sequence of steps based on some practical considerations, e.g., we will use the **Autoreject function** as a tool for automated artifact rejection and correction (Jas et al., 2017) to ensure data-driven and reproducible artifact rejection in a semi-automated preprocessing pipeline.



__Preprocessing sequence__ <br>
1. **Bad channel interpolation** goes first because noisy channels would contaminate all subsequent processing steps.
2. **Filtering (high-pass, low-pass)** is a pre-requisite for fitting ICA. By doing it before epoching, we avoid edge artifacts.
3. **Downsampling** reduces computational costs of ICA and Autoreject.
4. **Artefact correction with ICA** requires continuous data to identify stable components.
5. **Epoching (separately for encoding and retrieval)** is performed on clean continuous signal.
6. **Merge epochs & behavioral data frames** so that trial-level behavioral metadata is kept intact before any epochs get dropped.
7. **Autoreject** runs on epochs (not continuous data) to determine optimal peak-to-peak thresholds for the epochs. 
8. **Baseline correction** runs after Autoreject to ensure that bad channels and trials do not bias the baseline estimate.
9. **Re-referencing to electrode average** is done last, because it would be affected by any bad channels or epochs present in the data.

__Which steps belong to which stage__ <br>
- **Stage A, continuous data** (`preprocess_continuous`): steps 1-4 above (bad channel interpolation, filtering, downsampling, ICA). This is the slow part (about 3 minutes per participant) and identical for all analyses, so it is run once and the result is saved as `sub-XX_task-loc_desc-clean_raw.fif`.
- **Stage B, epochs** (`make_epochs`): steps 5-9 (epoching, merging behavioral data, Autoreject, baseline correction, re-referencing). The epoch settings come from `EPOCH_SPECS`.

__What is different for the ERS epochs__ <br>
- **Time lock:** the encoding epochs are locked to the *cue* onset (trigger `enc_cue`, code 101) instead of the target, and span the whole encoding stream (-0.2 to 7.0 s; the response prompt appears at 7.18 s or later). The retrieval epochs span the 2 s response-free cue period (-0.2 to 2 s).
- **Timing check:** every target is matched to the cue presented about 5.2 s earlier. The measured cue-to-target lag is written to the log, and a warning is logged if it deviates from 5.2 s by more than 100 ms (loose sanity check). The lag is not the nominal 5 s: in the PsychoPy logs of four participants every panel stays on screen 50-60 ms longer than programmed, so panel boundaries (2, 3, 4, 5 s) are only approximate. The epochs themselves are locked to the cue trigger and do not depend on this.
- **Baseline:** -0.2 to 0 s in both phases. Panels are not baseline-corrected separately, because there is no fixation between the panels of the encoding stream.
- **Autoreject:** runs on the long epochs, so more epochs are expected to be dropped than for the ERP epochs. Check `autoreject_pct_dropped` in the log, since a hit needs both its encoding and its retrieval epoch.

### 2a. Stage A: continuous preprocessing

Steps 1-4 of the pipeline. The function loads and crops the raw data, filters, interpolates bad channels, downsamples and applies the ICA correction, and saves the cleaned continuous recording.

In [2]:
# ==========================================
# 2a. STAGE A: CONTINUOUS PREPROCESSING (run once per subject)
# ==========================================
def preprocess_continuous(
    subject_id,
    bad_channels,
    bad_ics,
    l_freq=0.1, # This is in line with Gian's feedback (calculate ICA with 1 Hz HPF, apply to 0.1 HPF dataset)
    h_freq=40.0,
    resample_sfreq=250.0,
    buffer_sec = 20.0,
    desc="clean",
    overwrite=False,
):
    """
    Run steps 1-4 of the preprocessing pipeline (everything that happens on continuous data)
    for one subject and save the cleaned continuous recording.

    The saved file is the shared starting point for ALL epoch sets (ERP and ERS), so the
    expensive steps (filtering, ICA) are only run once per subject.

    Parameters
    ----------
    subject_id : str
        BIDS subject label, e.g. "23".
    bad_channels : list of str
        Channels to mark as bad and interpolate, from notebook 2's QC.
    bad_ics : list of int
        ICA component indices to exclude, from notebook 2's QC.
    desc : str
        Label in the output file name (sub-XX_task-loc_desc-<desc>_raw.fif). Use a different label
        (e.g. "clean_hp0.5" together with l_freq=0.5) for a sensitivity version with another high-pass.
    overwrite : bool
        If False and the output file already exists, skip this subject
        (idempotent re-runs — safe to re-execute the batch loop).

    Returns
    -------
    dict
        Summary log (also saved as JSON) with key metrics for this subject.
    """

    # Create various file paths
    subj_deriv_dir = derivatives_dir / f"sub-{subject_id}" / "eeg"
    plots_dir = subj_deriv_dir / "qc_plots"
    log_path = subj_deriv_dir / f"sub-{subject_id}_preproc_log_continuous_{desc}.json"
    clean_raw_path = subj_deriv_dir / f"sub-{subject_id}_task-loc_desc-{desc}_raw.fif"

    # --- Idempotency check: skip subject if the cleaned continuous data already exist ---
    if not overwrite and clean_raw_path.exists():
        print(f"sub-{subject_id}: {clean_raw_path.name} already exists, skipping (overwrite=False).")
        with open(log_path, 'r') as f:
            return json.load(f)

    log = {"subject": subject_id, "desc": desc, "steps_completed": [], "warnings": []}

    # ==========================================
    # STEP 1: LOAD RAW + APPLY BAD CHANNELS
    # ==========================================
    bids_path = BIDSPath(subject=subject_id, task='loc', datatype='eeg', root=bids_root)
    raw = read_raw_bids(bids_path=bids_path, verbose='error')
    raw.load_data()
    raw.info['bads'] = bad_channels
    log["bad_channels"] = bad_channels
    log["steps_completed"].append("load_raw")

    # ==========================================
    # STEP 1b: CROP TO TASK-RELEVANT SEGMENTS (encoding + retrieval)
    # ==========================================
    # Drops irrelevant recorded phases (e.g., instructions, breaks, ...).
    # Encoding and retrieval are cropped as separate continuous chunks (each with a buffer) 
    # and concatenated BEFORE filtering — same buffer size as used in notebook 2, 
    # so the QC inspection and the actual preprocessing operate on matching data.

    def get_onset(raw, description):
        onset = raw.annotations.onset[raw.annotations.description == description]
        if len(onset) == 0:
            raise ValueError(f"No annotation '{description}' found for sub-{subject_id}")
        return onset[0]

    enc_start = get_onset(raw, "enc_start")
    enc_end   = get_onset(raw, "enc_end")
    ret_start = get_onset(raw, "ret_start")
    ret_end   = get_onset(raw, "ret_end")

    t_min, t_max = raw.times[0], raw.times[-1]
    enc_tmin = max(enc_start - buffer_sec, t_min)
    enc_tmax = min(enc_end + buffer_sec, t_max)
    ret_tmin = max(ret_start - buffer_sec, t_min)
    ret_tmax = min(ret_end + buffer_sec, t_max)

    raw_enc = raw.copy().crop(tmin=enc_tmin, tmax=enc_tmax)
    raw_ret = raw.copy().crop(tmin=ret_tmin, tmax=ret_tmax)
    raw = mne.concatenate_raws([raw_enc, raw_ret])  # inserts boundary annotation at the seam

    log["crop_params"] = {
        "buffer_sec": buffer_sec,
        "encoding_window": [enc_tmin, enc_tmax],
        "retrieval_window": [ret_tmin, ret_tmax],
        "total_duration_sec": raw.times[-1],
    }
    log["steps_completed"].append("crop_concatenate")


    # ==========================================
    # STEP 2: FILTERING (low, high)
    # ==========================================

    # Plot PSD before filtering
    psd_before = raw.copy().pick('eeg').compute_psd(fmax=80)

    # Apply high pass and low pass filters
    raw.filter(l_freq=l_freq, h_freq=h_freq, verbose=False)
    log["filter_params"] = {"l_freq": l_freq, "h_freq": h_freq}
    log["steps_completed"].append("filter")

    # Plot PSD after filtering
    psd_after = raw.copy().pick('eeg').compute_psd(fmax=60)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
    psd_before.plot(axes=axes[0], show=False)
    axes[0].set_title("Before filter")
    psd_after.plot(axes=axes[1], show=False)
    axes[1].set_title("After filter")
    save_fig(fig, plots_dir, "01_psd_before_after_filter.png")


    # Interpolate bad channels now (before ICA apply / epoching)
    if bad_channels:
        raw.interpolate_bads(reset_bads=True, verbose=False)
        log["steps_completed"].append("interpolate_bads")

    # ==========================================
    # STEP 3: DOWNSAMPLE
    # ==========================================
    raw.resample(resample_sfreq, verbose=False)
    log["resample_sfreq"] = resample_sfreq
    log["steps_completed"].append("resample")

    # ==========================================
    # STEP 4: APPLY ICA (using bad_ics from notebook 2)
    # ==========================================
    # #ICA.apply() needs the actual unmixing matrix, which was saved in the previous notebook
    ica_path = derivatives_dir / f"sub-{subject_id}" / "eeg" / "ica_qc" / f"sub-{subject_id}_ica.fif"
    if not ica_path.exists():
        raise FileNotFoundError(
            f"No saved ICA solution found for sub-{subject_id} at {ica_path}. "
            f"Save ica.save(...) at the end of notebook 2."
        )
    ica = mne.preprocessing.read_ica(ica_path)
    ica.exclude = bad_ics
    ica.apply(raw)
    log["excluded_ics"] = bad_ics
    log["steps_completed"].append("ica_apply")

    # ==========================================
    # STEP 4b: SAVE THE CLEANED CONTINUOUS DATA
    # ==========================================
    # This is the hand-over point to Stage B (epoching). The event annotations (triggers) are stored in the
    # .fif file, so Stage B can extract events from it exactly like before.
    subj_deriv_dir.mkdir(parents=True, exist_ok=True)
    raw.save(clean_raw_path, overwrite=True)
    log["steps_completed"].append("save_clean_raw")

    with open(log_path, 'w') as f:
        json.dump(log, f, indent=2)

    print(f"sub-{subject_id}: continuous preprocessing complete -> {clean_raw_path.name}")

    return log

### 2b. Epoch specifications

The settings that differ between analyses are collected in one dictionary, so that the ERP and the ERS epochs are created by the same function.

In [3]:
# ==========================================
# 2b. EPOCH SPECIFICATIONS
# ==========================================
# One entry per analysis. 'anchor' says which event the epochs are time-locked to:
#   encoding : 'target' = target-panel onset (ERP analysis), 'cue' = cue-panel onset (ERS analysis)
#   retrieval: always the retrieval cue (no 'anchor' needed)
# 'suffix' is appended to the 'desc' label of the output files. The ERP spec has no suffix, so the ERP epochs keep
# their original file names (desc-encoding / desc-retrieval).
EPOCH_SPECS = {
    "erp": {
        "suffix": "",
        "encoding":  dict(anchor="target", tmin=-0.5, tmax=1.0, baseline=(None, 0)),
        "retrieval": dict(tmin=-0.5, tmax=1.0, baseline=(None, 0)),
    },
    "ers": {
        "suffix": "ERS",
        # Encoding epoch spans the whole encoding stream, retrieval epoch the 2 s response-free cue period.
        # Same short baseline in both phases (the retrieval fixation is 2 s, the encoding fixation 0.5 s, so only the
        # last 200 ms before the cue are used).
        #
        # Timing: nominal panel durations (cue 2 s, three actions 1 s each, target 2 s). In the PsychoPy logs of four
        # participants every panel was shown 50-60 ms longer than programmed (cue-to-target lag about 5.2 s instead of 5 s),
        # so panel boundaries are only approximate. The epochs are locked to the enc_cue trigger and do not depend on this.
        # tmax = 7.0 s ends before the response prompt appears (earliest 7.18 s) and cuts off only the last ~0.2 s of the target.
        "encoding":  dict(anchor="cue", tmin=-0.2, tmax=7.0, baseline=(-0.2, 0),
                          cue_to_target_sec=5.2),  # approximate cue-to-target lag (see above); used for matching + as a loose sanity check
        "retrieval": dict(tmin=-0.2, tmax=2.0, baseline=(-0.2, 0)),
    },
}

### 2c. Stage B: epoching, Autoreject and re-referencing

Steps 5-7 of the pipeline, run on the cleaned continuous data for one epoch specification.

In [4]:
# ==========================================
# 2c. STAGE B: EPOCHING + AUTOREJECT + RE-REFERENCING (run once per epoch specification)
# ==========================================
def make_epochs(
    subject_id,
    spec_name="ers",
    clean_desc="clean",
    enc_cue_name="enc_cue",
    cue_lag_tolerance=0.1,
    overwrite=False,
):
    """
    Run steps 5-7 of the preprocessing pipeline (epoching, Autoreject, average reference) on the cleaned
    continuous data of one subject, following the epoch specification EPOCH_SPECS[spec_name].

    Parameters
    ----------
    subject_id : str
        BIDS subject label, e.g. "23".
    spec_name : str
        Key of EPOCH_SPECS: "erp" (target-locked encoding epochs, as before) or "ers" (cue-locked epochs
        covering the entire encoding stream and the retrieval cue period).
    clean_desc : str
        desc label of the cleaned continuous file produced by preprocess_continuous().
    enc_cue_name : str
        Annotation name of the encoding cue trigger (trigger code 101).
    cue_lag_tolerance : float
        Maximum allowed deviation (s) of the measured cue-to-target lag from the expected lag (cue_to_target_sec in the
        spec, about 5.2 s). The measured lags of the four checked participants lie within 5.19-5.28 s. If it is violated,
        a warning is logged (see log["warnings"]).
    overwrite : bool
        If False and output files already exist, skip this subject.

    Returns
    -------
    dict
        Summary log (also saved as JSON) with key metrics for this subject.
    """

    spec = EPOCH_SPECS[spec_name]
    suffix = spec["suffix"]

    # Create various file paths
    subj_deriv_dir = derivatives_dir / f"sub-{subject_id}" / "eeg"
    plots_dir = subj_deriv_dir / "qc_plots"
    clean_raw_path = subj_deriv_dir / f"sub-{subject_id}_task-loc_desc-{clean_desc}_raw.fif"
    # The ERP spec keeps the original log file name, other specs get their own log
    log_path = subj_deriv_dir / (f"sub-{subject_id}_preproc_log.json" if not suffix
                                 else f"sub-{subject_id}_preproc_log_{spec_name}.json")
    beh_path = bids_root / f"sub-{subject_id}" / "beh" / f"sub-{subject_id}_task-loc_label-merged_beh.csv"
    epochs_enc_path = subj_deriv_dir / f"sub-{subject_id}_task-loc_desc-encoding{suffix}_epo.fif"
    epochs_ret_path = subj_deriv_dir / f"sub-{subject_id}_task-loc_desc-retrieval{suffix}_epo.fif"

    # --- Idempotency check: skip subject if these epochs were already created ---
    if not overwrite and epochs_enc_path.exists() and epochs_ret_path.exists():
        print(f"sub-{subject_id}: {spec_name} epochs already exist, skipping (overwrite=False).")
        with open(log_path, 'r') as f:
            return json.load(f)

    log = {"subject": subject_id, "spec": spec_name, "steps_completed": [], "warnings": []}

    # --- Load the cleaned continuous data from Stage A ---
    if not clean_raw_path.exists():
        raise FileNotFoundError(
            f"No cleaned continuous data for sub-{subject_id} at {clean_raw_path}. "
            f"Run preprocess_continuous() first."
        )
    raw = mne.io.read_raw_fif(clean_raw_path, preload=True, verbose='error')
    log["steps_completed"].append("load_clean_raw")

    # ==========================================
    # STEP 5: EPOCHING (encoding + retrieval, separately)
    # ==========================================

    # --- Load and prepare behavioral data (encoding and retrival are merged already) ---
    beh_path = (bids_root / f"sub-{subject_id}" / "beh"
                / f"sub-{subject_id}_task-loc_label-merged_beh.csv")
    beh_df = pd.read_csv(beh_path)

    # This function assigns each trial a label for the schema and perceptual condition (either SC or SI, and PC or PI)
    def sc_si_pc_pi(row):
        sc_si = 'SI' if row['enc_high_prediction'] == 1 else 'SC'
        pc_pi = 'PI' if row['enc_low_prediction'] == 1 else 'PC'
        return sc_si, pc_pi

    # We create a dataframe that only contains the 64 encoding trials
    beh_enc = (beh_df.dropna(subset=['enc_trial_count']) # Drops rows that have NULL values in column 'enc_trial_count'
                      .sort_values('enc_trial_count') # Sort encoding trials by trial_count
                      .reset_index(drop=True)) # Discard old indices and sets new 0-index ascending by trial_count

    # Function to assign each row of the encoding data a condition label
    # in the form of < "Target" / "SC" or "SI" / "PC" or "PI" >
    def enc_label(r):
        return 'Target/' + '/'.join(sc_si_pc_pi(r))

    beh_enc['condition_label'] = beh_enc.apply(enc_label, axis=1)

    # Similarly, we create a retrieval dataframe containing 84 rows, in their retrieval presentation order
    beh_ret = beh_df.sort_values('ret_trial_count').reset_index(drop=True)

    def ret_label(row):
        if row['ret_trial_type'] == 'new':
            return 'Cue/New'
        return 'Cue/Old/' + '/'.join(sc_si_pc_pi(row))
    beh_ret['condition_label'] = beh_ret.apply(ret_label, axis=1)

    # Store the behavioral data for the two phases in a dictionairy
    beh_metadata = {'encoding': beh_enc, 'retrieval': beh_ret}

    # --- Extract events from annotations ---
    events, event_id = mne.events_from_annotations(raw, verbose=False)

    # Get unique event codes actually present in this recording 
    present_ids = set(events[:, 2])

    # If we want to include the participants with the behavioral bug, we'd need to add a 'bug' condition here 
    enc_target_event_ids = {
        'Target/SC/PC': event_id['tgt_sc_pc'],
        'Target/SC/PI': event_id['tgt_sc_pi'],
        'Target/SI/PC': event_id['tgt_si_pc'],
        'Target/SI/PI': event_id['tgt_si_pi'],
    }

    ret_cue_event_ids = {
        'Cue/Old/SC/PC': event_id['ret_cue_old_sc_pc'],
        'Cue/Old/SC/PI': event_id['ret_cue_old_sc_pi'],
        'Cue/Old/SI/PC': event_id['ret_cue_old_si_pc'],
        'Cue/Old/SI/PI': event_id['ret_cue_old_si_pi'],
        'Cue/New': event_id['ret_cue_new'],
    }

    # Collect epochs by iterating twice: once with phase 'encoding', once with phase ' retrieval'
    epochs_dict = {}
    for phase, phase_event_ids in [('encoding', enc_target_event_ids),
                                    ('retrieval', ret_cue_event_ids)]:

        # Dict comprehension: we store every pair of label and event ID (label, eid) in phase_event_ids as a new dict entry IF
        # that eid is in present_id.
        available = {label: eid for label, eid in phase_event_ids.items()
                     if eid in present_ids}

        # Check for missing event IDs in a phase
        missing = set(phase_event_ids) - set(available)
        if missing:
            log["warnings"].append(
                f"{phase}: missing conditions in recording — {sorted(missing)}"
            )

        # Check if a phase is wholly missing
        if not available:
            log["warnings"].append(f"No events found for {phase} phase — skipped.")
            continue

        # Filter the full events array down to just this phase's matching triggers,
        # in chronological order — this is what metadata needs to align against.
        phase_mask = np.isin(events[:, 2], list(available.values())) # Boolean masking: vector with length of nrows[events], TRUE if the event code
        # is in our dictionairy of event codes we care about (available)
        phase_events = events[phase_mask] # This contains only encoding (or retrieval events) in original chronological order

        # Settings of this phase from the epoch specification
        ps = spec[phase]

        # --- Which events do we epoch on? ---
        # Default: the events we just extracted (encoding: target onsets, retrieval: cue onsets).
        # For cue-locked encoding epochs (ERS analysis) we epoch on the encoding cue trigger instead. We match every
        # target to the cue that was presented cue_to_target_sec (about 5.2 s, measured in the PsychoPy logs) earlier. This (1) makes sure that cue i and target i
        # belong to the same trial (also if there should be additional cue triggers, e.g., from practice trials) and
        # (2) lets us verify the timing between the two panels with the real triggers (the expected lag is about 5.2 s, not the
        # nominal 5 s, because each panel is shown 50-60 ms longer than nominal; used as a loose sanity check only).
        # phase_events (targets) is still needed below, because only the target triggers carry the condition labels.
        epoch_events = phase_events
        epoch_event_id = available
        if phase == 'encoding' and ps['anchor'] == 'cue':
            sfreq = raw.info['sfreq']
            if enc_cue_name not in event_id:
                log["warnings"].append(f"{enc_cue_name} not found in annotations - cue onset derived from target onset.")
                epoch_events = phase_events.copy()
                epoch_events[:, 0] -= int(round(ps['cue_to_target_sec'] * sfreq))
            else:
                cue_events = events[events[:, 2] == event_id[enc_cue_name]]
                expected_cue_samples = phase_events[:, 0] - int(round(ps['cue_to_target_sec'] * sfreq))
                # For every target, find the cue trigger closest to the expected cue time (|n_targets| x |n_cues| matrix)
                nearest = np.abs(cue_events[:, 0][None, :] - expected_cue_samples[:, None]).argmin(axis=1)
                lag = (phase_events[:, 0] - cue_events[nearest, 0]) / sfreq  # measured cue-to-target lag in seconds
                log["enc_cue_to_target_lag_sec"] = {"mean": float(lag.mean()), "min": float(lag.min()), "max": float(lag.max())}
                if np.abs(lag - ps['cue_to_target_sec']).max() > cue_lag_tolerance:
                    log["warnings"].append(
                        f"encoding: cue-to-target lag deviates from {ps['cue_to_target_sec']} s by more than "
                        f"{cue_lag_tolerance} s (min {lag.min():.3f}, max {lag.max():.3f}) - check trigger timing."
                    )
                if len(cue_events) != len(phase_events):
                    log["warnings"].append(
                        f"encoding: {len(cue_events)} cue triggers vs. {len(phase_events)} target triggers "
                        f"(matched by timing)."
                    )
                epoch_events = cue_events[nearest]
                epoch_event_id = {enc_cue_name: event_id[enc_cue_name]}

        metadata = beh_metadata[phase]

        # Comparing the length of behavioral data to the length of extracted events for a given phase.
        # This check currently fails the subjects with the bug in the task code.
        if len(metadata) != len(phase_events):
            log["warnings"].append(
                f"{phase}: behavioral rows ({len(metadata)}) != triggered EEG events "
                f"({len(phase_events)}) — metadata NOT attached for sub-{subject_id}, check alignment."
            )
            metadata_to_use = None
        else:
            metadata_to_use = metadata

        # Building the epochs
        epochs = mne.Epochs(
            raw, epoch_events, event_id=epoch_event_id,
            tmin=ps['tmin'], tmax=ps['tmax'], baseline=ps['baseline'],
            preload=True, reject_by_annotation=True, verbose=False,
            metadata=metadata_to_use,
        )

        # Sanity check: do trigger-derived and CSV-derived labels agree for the same epoch?
        if metadata_to_use is not None:
            # Inversion of keys and values of available: id_to_label now maps code to label
            id_to_label = {v: k for k, v in available.items()}
            # List comprehension: for every trigger code actually present in the final epochs object, 
            # look up its label. Result: a list of condition labels, in the same order as the epochs.
            # (Changed from before: we look the labels up via epochs.selection, i.e., the rows of phase_events that
            # survived epoching. That gives the same result as before for target/cue-locked epochs, and also works
            # if the epochs are locked to a different trigger than the one carrying the condition label.)
            label_by_row = [id_to_label[code] for code in phase_events[:, 2]]
            triggered_labels = [label_by_row[i] for i in epochs.selection]

            # Converts the dataframe column to a Python list, to allow comparison to triggered_labels
            beh_labels = epochs.metadata['condition_label'].tolist()

            # Create a list of indices, where the pair of (a) triggered_labels[i] and beh_labels [i] does not line up
            mismatches = [i for i, (a, b) in enumerate(zip(triggered_labels, beh_labels)) if a != b]
            if mismatches:
                log["warnings"].append(
                    f"{phase}: {len(mismatches)} trigger/CSV condition mismatches "
                    f"(first at epoch index {mismatches[0]}) — inspect trial order."
                )

        epochs_dict[phase] = epochs

    log["steps_completed"].append("epoching")
    log["n_epochs_pre_autoreject"] = {k: len(v) for k, v in epochs_dict.items()}
    # ==========================================
    # STEP 6: AUTOREJECT (separately per phase)
    # ==========================================
    # The Autoreject function uses cross-validation to identify an optimal peak-to-peak threshold for a given dataset
    # Based on it, bad channels are either interpolated or epochs are dropped
    epochs_clean = {}
    for phase, epochs in epochs_dict.items():
        # Evoked BEFORE autoreject
        fig_evoked_before = epochs.average().plot(show=False)
        save_fig(fig_evoked_before, plots_dir, f"02_evoked_{phase}{suffix}_before_autoreject.png")

        ar = AutoReject(
            random_state=97,  # random seed
            n_jobs=1, # use 1 CPU core
            verbose=False
            ) 
        epochs_ar, reject_log = ar.fit_transform(epochs, return_log=True)
        epochs_clean[phase] = epochs_ar

        # Channel-level summary of the reject log (labels: 0 = good, 1 = bad, 2 = interpolated), per epoch
        n_bad_ch = (reject_log.labels == 1).sum(axis=1)
        n_int_ch = (reject_log.labels == 2).sum(axis=1)
        log.setdefault("autoreject_channels_per_epoch", {})[phase] = {
            "bad_mean": float(n_bad_ch.mean()), "interpolated_mean": float(n_int_ch.mean()),
            "epochs_with_more_than_10pct_bad": int((n_bad_ch > 0.1 * reject_log.labels.shape[1]).sum()),
        }

        log.setdefault("n_epochs_post_autoreject", {})[phase] = len(epochs_ar)
        log.setdefault("autoreject_pct_dropped", {})[phase] = round(
            100 * (1 - len(epochs_ar) / len(epochs)), 1
        )

        # NEW: per-condition breakdown, same spot as the other post-AR counts
        log.setdefault("n_epochs_per_condition", {})[phase] = (
            epochs_ar.metadata['condition_label'].value_counts().to_dict()
            if epochs_ar.metadata is not None else {}
        )
        # Evoked AFTER autoreject
        fig_evoked_after = epochs_ar.average().plot(show=False)
        save_fig(fig_evoked_after, plots_dir, f"02_evoked_{phase}{suffix}_after_autoreject.png")

        # Reject log visualization (which epochs/channels were dropped/interpolated)
        fig_reject = reject_log.plot(show=False)
        save_fig(fig_reject, plots_dir, f"05_autoreject_log_{phase}{suffix}.png")

    
    log["steps_completed"].append("autoreject")


    # ==========================================
    # STEP 7: AVERAGE RE-REFERENCING
    # ==========================================
    for phase, epochs in epochs_clean.items():
        epochs.set_eeg_reference('average', projection=False, verbose=False)
    log["steps_completed"].append("average_reference")

    # ==========================================
    # STEP 8: SAVE OUTPUTS
    # ==========================================
    subj_deriv_dir.mkdir(parents=True, exist_ok=True)
    epochs_clean['encoding'].save(epochs_enc_path, overwrite=overwrite) if 'encoding' in epochs_clean else None
    epochs_clean['retrieval'].save(epochs_ret_path, overwrite=overwrite) if 'retrieval' in epochs_clean else None

    with open(log_path, 'w') as f:
        json.dump(log, f, indent=2)

    print(f"sub-{subject_id} [{spec_name}]: epoching complete. "
          f"Encoding epochs: {log.get('n_epochs_post_autoreject', {}).get('encoding', 'N/A')}, "
          f"Retrieval epochs: {log.get('n_epochs_post_autoreject', {}).get('retrieval', 'N/A')}")

    return log

### 3. Run preprocessing function

#### 3.1 Single subject

Use this cell if you want to test the output of the preprocessing function for a single subject, e.g., to debug and troubleshoot.

In [5]:
# Stage A once, then Stage B for each epoch specification
test_log = preprocess_continuous(
    subject_id="09",
    bad_channels=["C2", "CP1"],
    bad_ics=[0, 1, 4, 7, 11, 12, 15, 17, 18, 19, 21, 22, 24, 25, 27, 28, 31, 32, 33],
    overwrite=True,
)

test_log_erp = make_epochs(subject_id="09", spec_name="erp", overwrite=True)
test_log_ers = make_epochs(subject_id="09", spec_name="ers", overwrite=True)

# Check the warnings and the timing between the encoding cue and target trigger (should be ~5.0 s), and how many
# epochs survive Autoreject in the (much longer) ERS epochs
print(test_log_ers["warnings"])
print(test_log_ers.get("enc_cue_to_target_lag_sec"))
print(test_log_ers["n_epochs_post_autoreject"], test_log_ers["autoreject_pct_dropped"])

Reading 0 ... 3087499  =      0.000 ...  3087.499 secs...
Effective window size : 2.048 (s)
Effective window size : 2.048 (s)
Plotting power spectral density (dB=True).
Plotting power spectral density (dB=True).
Reading c:\Users\noahm\projects\loc_analysis\data\derivatives\sub-09\eeg\ica_qc\sub-09_ica.fif ...
Now restoring ICA solution ...
Ready.
Applying ICA to Raw instance
    Transforming to ICA space (60 components)
    Zeroing out 19 ICA components
    Projecting back using 61 PCA components
Overwriting existing file.
Writing c:\Users\noahm\projects\loc_analysis\data\derivatives\sub-09\eeg\sub-09_task-loc_desc-clean_raw.fif
Overwriting existing file.
Closing c:\Users\noahm\projects\loc_analysis\data\derivatives\sub-09\eeg\sub-09_task-loc_desc-clean_raw.fif
[done]
sub-09: continuous preprocessing complete -> sub-09_task-loc_desc-clean_raw.fif
Dropped 4 epochs: 44, 52, 57, 59
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.


#### 3.2 Batch preprocessing

Run the cells below if you want to run the pre-processing for all subjects that underwent the BIDS conversion and were visually inspected and configured in `2_manual_qc_and_config.ipynb`. Right now (July 22nd, 2026), this amounts to 18 participants.

The cell loads the configuration CSV file into a dataframe, which includes the subject numbers as an identifier, plus information about the integrity of the data recordings (EEG, behavioral) and importantly, the channels selected for interpolation aswell as the independent components to use for artefact correction.

In [7]:
# We loaded df_config in the set-up cell already

# Retrieve indices of subjects, that (1) are not behaviorally excluded &
# (2) have complete EEG recordings for encoding and retrieval phases

ready_mask = (
    (~df_config['is_excluded'].astype(bool)) &
    (df_config['eeg_enc_recorded'] == True) &
    (df_config['eeg_ret_recorded'] == True)
)
subjects_to_process = df_config[ready_mask]

print(f"{len(subjects_to_process)} of {len(df_config)} subjects ready for batch preprocessing.")

22 of 22 subjects ready for batch preprocessing.


This cell performs the actual preprocessing, looping over the selected participants.
Be careful! This can take a while to run. For me, about three minutes per participant. 
With the two-stage setup, Stage A takes about three minutes per participant (skipped if the cleaned file already exists); Stage B for the ERP and the ERS epochs adds the time for Autoreject, which is longer for the 7 s ERS epochs.

In [8]:
batch_summary = []
for _, row in subjects_to_process.iterrows():
    subj_id = row['subject']
    bad_chs = [c.strip() for c in row['bad_channels'].split(',')] if pd.notna(row['bad_channels']) and row['bad_channels'] else []
    bad_ic_list = [int(x.strip()) for x in row['bad_icas'].split(',')] if pd.notna(row['bad_icas']) and row['bad_icas'] else []

    try:
        # Stage A: continuous data (slow, ~3 min; skipped if the clean file already exists)
        preprocess_continuous(
            subject_id=subj_id,
            bad_channels=bad_chs,
            bad_ics=bad_ic_list,
            overwrite=False,  # skip subjects already processed
        )
        # Stage B: epochs for the ERP and the ERS analysis
        log_erp = make_epochs(subject_id=subj_id, spec_name="erp", overwrite=False)
        log_ers = make_epochs(subject_id=subj_id, spec_name="ers", overwrite=False)
        batch_summary.append({
            "subject": subj_id, "status": "success",
            **{f"erp_{k}": v for k, v in log_erp.get("n_epochs_post_autoreject", {}).items()},
            **{f"ers_{k}": v for k, v in log_ers.get("n_epochs_post_autoreject", {}).items()},
            "ers_warnings": "; ".join(log_ers.get("warnings", [])),
        })
    except Exception as e:
        print(f"sub-{subj_id}: FAILED — {e}")
        batch_summary.append({"subject": subj_id, "status": "failed", "error": str(e)})

df_batch_summary = pd.DataFrame(batch_summary)
df_batch_summary.to_csv(derivatives_dir / "batch_preprocessing_summary.csv", index=False)
display(df_batch_summary)

Reading 0 ... 3611599  =      0.000 ...  3611.599 secs...
Effective window size : 2.048 (s)
Effective window size : 2.048 (s)
Plotting power spectral density (dB=True).
Plotting power spectral density (dB=True).
Reading c:\Users\noahm\projects\loc_analysis\data\derivatives\sub-17\eeg\ica_qc\sub-17_ica.fif ...
Now restoring ICA solution ...
Ready.
Applying ICA to Raw instance
    Transforming to ICA space (61 components)
    Zeroing out 17 ICA components
    Projecting back using 62 PCA components
Writing c:\Users\noahm\projects\loc_analysis\data\derivatives\sub-17\eeg\sub-17_task-loc_desc-clean_raw.fif
Closing c:\Users\noahm\projects\loc_analysis\data\derivatives\sub-17\eeg\sub-17_task-loc_desc-clean_raw.fif
[done]
sub-17: continuous preprocessing complete -> sub-17_task-loc_desc-clean_raw.fif
sub-17: erp epochs already exist, skipping (overwrite=False).
Dropped 9 epochs: 0, 2, 3, 7, 8, 17, 27, 48, 56
sub-17 [ers]: epoching complete. Encoding epochs: 55, Retrieval epochs: 84
Reading 0 

,subject,status,erp_encoding,erp_retrieval,ers_encoding,ers_retrieval,ers_warnings
0,17,success,62,84,55,84,
1,20,success,64,80,64,82,retrieval: 64 trigger/CSV condition mismatches...
2,21,success,44,51,35,84,retrieval: 64 trigger/CSV condition mismatches...
3,22,success,61,84,64,84,retrieval: 64 trigger/CSV condition mismatches...
4,23,success,64,84,64,84,retrieval: 64 trigger/CSV condition mismatches...
5,26,success,64,81,60,81,
6,27,success,63,74,64,73,encoding: cue-to-target lag deviates from 5.2 ...
7,30,success,62,83,64,84,
8,32,success,64,84,62,79,
9,33,success,64,82,64,84,


In [13]:
import mne, pandas as pd

def seqs(sub):
    raw = mne.io.read_raw_fif(derivatives_dir / f"sub-{sub}" / "eeg" / f"sub-{sub}_task-loc_desc-clean_raw.fif",
                              preload=False, verbose="error")
    ev, eid = mne.events_from_annotations(raw, verbose=False); inv = {v: k for k, v in eid.items()}
    names = [inv[c] for c in ev[:, 2]]
    trig_enc = [n for n in names if n.startswith("tgt_")]
    trig_ret = [n for n in names if n.startswith("ret_cue_")]
    beh = pd.read_csv(bids_root / f"sub-{sub}" / "beh" / f"sub-{sub}_task-loc_label-merged_beh.csv")
    sc = lambda r: ("si" if r.enc_high_prediction == 1 else "sc", "pi" if r.enc_low_prediction == 1 else "pc")
    be = beh.dropna(subset=["enc_trial_count"]).sort_values("enc_trial_count")
    csv_enc = ["tgt_%s_%s" % sc(r) for r in be.itertuples()]
    br = beh.sort_values("ret_trial_count")
    csv_ret = ["ret_cue_new" if r.ret_trial_type == "new" else "ret_cue_old_%s_%s" % sc(r) for r in br.itertuples()]
    return trig_enc, csv_enc, trig_ret, csv_ret

def compare(trig, csv, name):
    n = min(len(trig), len(csv)); mism = [i for i in range(n) if trig[i] != csv[i]]
    print(f"{name}: {len(trig)} Trigger, {len(csv)} CSV-Zeilen, {len(mism)} Mismatches, erste Indizes {mism[:10]}")
    for s in range(-3, 4):   # konstante Verschiebung (fehlender/zusätzlicher Trigger)?
        a, b = trig[max(s, 0):], csv[max(-s, 0):]; m = min(len(a), len(b))
        print(f"   Shift {s:+d}: {sum(x == y for x, y in zip(a[:m], b[:m]))} von {m} passen")

te, ce, tr, cr = seqs("23")   # betroffene Subjekt-ID einsetzen
compare(te, ce, "encoding"); compare(tr, cr, "retrieval")

encoding: 64 Trigger, 64 CSV-Zeilen, 0 Mismatches, erste Indizes []
   Shift -3: 9 von 61 passen
   Shift -2: 17 von 62 passen
   Shift -1: 17 von 63 passen
   Shift +0: 64 von 64 passen
   Shift +1: 17 von 63 passen
   Shift +2: 17 von 62 passen
   Shift +3: 9 von 61 passen
retrieval: 84 Trigger, 84 CSV-Zeilen, 64 Mismatches, erste Indizes [0, 1, 2, 4, 5, 6, 8, 9, 11, 14]
   Shift -3: 19 von 81 passen
   Shift -2: 20 von 82 passen
   Shift -1: 15 von 83 passen
   Shift +0: 20 von 84 passen
   Shift +1: 18 von 83 passen
   Shift +2: 19 von 82 passen
   Shift +3: 17 von 81 passen


In [14]:
for sub in ["23", "10"]:
    te, ce, tr, cr = seqs(sub)
    print(sub)
    print(pd.crosstab(pd.Series(tr, name="Retrieval-Trigger"), pd.Series(cr, name="CSV (= Encoding-Trigger)")))

23
CSV (= Encoding-Trigger)  ret_cue_new  ret_cue_old_sc_pc  ret_cue_old_sc_pi  \
Retrieval-Trigger                                                             
ret_cue_new                        20                  0                  0   
ret_cue_old_sc_pc                   0                  0                  0   
ret_cue_old_sc_pi                   0                  0                  0   
ret_cue_old_si_pc                   0                 16                  0   
ret_cue_old_si_pi                   0                  0                 16   

CSV (= Encoding-Trigger)  ret_cue_old_si_pc  ret_cue_old_si_pi  
Retrieval-Trigger                                               
ret_cue_new                               0                  0  
ret_cue_old_sc_pc                         0                 16  
ret_cue_old_sc_pi                        16                  0  
ret_cue_old_si_pc                         0                  0  
ret_cue_old_si_pi                         0          

In [ ]:
def check_alignment(sub):
    te, ce, tr, cr = seqs(sub)
    beh = pd.read_csv(bids_root / f"sub-{sub}" / "beh" / f"sub-{sub}_task-loc_label-merged_beh.csv").sort_values("ret_trial_count")
    ids = beh["enc_condition_id"].tolist()                      # NaN bei Foils
    old = [(t, int(i)) for t, i in zip(tr, ids) if t != "ret_cue_new" and pd.notna(i)]
    ct = pd.crosstab(pd.Series([t for t, _ in old], name="Trigger"), pd.Series([i for _, i in old], name="condition_id"))
    ok = len(old) == 64 and (ct > 0).sum(axis=0).max() == 1     # jede condition_id -> genau ein Trigger-Etikett
    return ok, ct

for sub in ["10", "11", "12", "20", "21", "22", "23", "27"]:        # hier alle 7 betroffenen Subjekt-IDs eintragen
    ok, ct = check_alignment(sub)
    print(sub, "eindeutig:", ok); print(ct)

10 eindeutig: True
condition_id       1  2  3  4  5  6  7  8
Trigger                                  
ret_cue_old_sc_pc  0  8  8  0  0  0  0  0
ret_cue_old_sc_pi  0  0  0  0  0  8  8  0
ret_cue_old_si_pc  0  0  0  8  8  0  0  0
ret_cue_old_si_pi  8  0  0  0  0  0  0  8
11 eindeutig: True
condition_id       1  2  3  4  5  6  7  8
Trigger                                  
ret_cue_old_sc_pc  0  0  8  8  0  0  0  0
ret_cue_old_sc_pi  0  0  0  0  0  0  8  8
ret_cue_old_si_pc  0  0  0  0  8  8  0  0
ret_cue_old_si_pi  8  8  0  0  0  0  0  0
12 eindeutig: True
condition_id       1  2  3  4  5  6  7  8
Trigger                                  
ret_cue_old_sc_pc  0  0  0  8  8  0  0  0
ret_cue_old_sc_pi  8  0  0  0  0  0  0  8
ret_cue_old_si_pc  0  0  0  0  0  8  8  0
ret_cue_old_si_pi  0  8  8  0  0  0  0  0
20 eindeutig: True
condition_id       1  2  3  4  5  6  7  8
Trigger                                  
ret_cue_old_sc_pc  0  0  0  8  8  0  0  0
ret_cue_old_sc_pi  8  0  0  0  0  0  0  8


## Note: warning "retrieval: trigger/CSV condition mismatches" (subjects 10, 11, 12, 20, 21, 22, 23)

**What the warning means.** For the retrieval epochs, `make_epochs` compares the condition names carried by the triggers (e.g. `Cue/Old/SC/PC`) with the condition from the behavioral CSV. For seven subjects these do not agree (32 triggers for subject 10, 64 for each of the others).

**Alignment check (`check_alignment`)**: establishes whether only the *names* of the triggers are wrong, or whether the *order* of trials is off:

1. The encoding triggers match the CSV for all subjects (64/64).
2. The positions of the foils in the retrieval sequence match as well.
3. For the old cues (64 trials), a crosstab is built: retrieval trigger label × `enc_condition_id` from the CSV.
4. The check passes (`eindeutig: True`) if there are exactly 64 old trials and every `enc_condition_id` maps to exactly one trigger label (8 trials each).

**Result.** The check passed for all seven subjects. The mapping `condition_id` → trigger label is therefore unique, but rotated differently per subject. The retrieval triggers consistently carry the wrong condition names, while the trial order is correct. Subject 27 shows the correct mapping (IDs 1,2 = SC/PC; 3,4 = SI/PC; 5,6 = SC/PI; 7,8 = SI/PI).

**Consequence.**
- No exclusion and no re-preprocessing.
- Always take conditions in retrieval epochs from the metadata (`enc_high_prediction`, `enc_low_prediction`, `condition_label`), never from the event names. Example: `epochs["enc_high_prediction == 0 and enc_low_prediction == 0"]`.
- H5 pairing (via `enc_trial_count`) and ERP analyses based on encoding triggers are not affected.
- Mention in the methods.